# SAM2 vs UltraSAM2 — Metric Comparison
Compares vanilla SAM2.1 (baseline) against UltraSAM2 (fine-tuned) under two input conditions:
- **Raw**: inference on original `cropped_pngs3` images
- **Denoised**: inference on Speckle2Self-denoised images (epoch 199)

Metrics loaded from pre-computed CSVs exported by each individual metrics notebook.

## Metric Guide

### IoU — Intersection over Union
Measures the overlap between the predicted mask and the ground-truth mask as a ratio of their intersection to their union. Ranges from 0 (no overlap) to 1 (perfect overlap). Also known as the Jaccard index. A score above 0.5 is generally considered acceptable; above 0.7 is good. Sensitive to large shape errors because the union grows quickly when predictions are over- or under-segmented.

### Dice — Dice Similarity Coefficient
Also called the F1 score for segmentation. Like IoU but gives more weight to the overlapping region: `Dice = 2·|A∩B| / (|A|+|B|)`. Ranges from 0 to 1. Always ≥ IoU for the same prediction. More forgiving than IoU for small missed regions, which makes it the preferred metric in medical image segmentation literature. A Dice of 0.80 roughly corresponds to an IoU of ~0.67.

### HD — Hausdorff Distance (pixels)
Measures boundary accuracy. Defined as the maximum of the minimum distances from each point on one contour to the nearest point on the other. Unlike IoU/Dice, which measure volumetric overlap, HD captures how far off the worst-case boundary deviation is. Lower is better; 0 means the contours match exactly. Very sensitive to outliers (a single stray pixel can inflate it), so it is reported alongside overlap metrics rather than as a standalone.

### mAP — mean Average Precision
Standard detection metric borrowed from COCO. For each IoU threshold in [0.50, 0.55, …, 0.95], the precision–recall curve is computed and the area under it (Average Precision, AP) is calculated; mAP is the mean across all thresholds. Ranges from 0 to 1. More demanding than a single-threshold metric because a prediction must hold up across a range of overlap requirements. A model that only just clears IoU=0.5 will score low here even if its Dice looks decent.

### mAP@50 — mean Average Precision at IoU ≥ 0.50
The same as AP but evaluated at a single IoU threshold of 0.50 — the traditional Pascal VOC criterion. Easier to pass than mAP (only requires 50 % overlap) and therefore typically the highest-scoring mAP variant. Useful as a headline detection number; compare it against mAP to gauge how sharply performance falls off as the overlap requirement tightens.

In [11]:
import os
import pandas as pd

In [12]:
# ── Load per-model CSVs ───────────────────────────────────────────────────────
sam2_raw = pd.read_csv(r"D:\sam2-results\sam2_metrics.csv",                          index_col=0)
sam2_dn  = pd.read_csv(r"D:\sam2-results\sam2_metrics_dn.csv",                       index_col=0)
us2_raw  = pd.read_csv(r"D:\dissertation-results\ultrasam2_metrics.csv",             index_col=0)
us2_dn   = pd.read_csv(r"D:\ultrasam2-results\ultrasam2_metrics_dn.csv",             index_col=0)

for name, df in [("SAM2 raw", sam2_raw), ("SAM2 dn", sam2_dn),
                 ("UltraSAM2 raw", us2_raw), ("UltraSAM2 dn", us2_dn)]:
    print(f"{name}: {list(df.index)}")

SAM2 raw: ['Patient 1', 'Patient 2', 'Patient 3', 'OVERALL']
SAM2 dn: ['Patient 1', 'Patient 2', 'Patient 3', 'OVERALL']
UltraSAM2 raw: ['Patient 1', 'Patient 2', 'Patient 3', 'OVERALL']
UltraSAM2 dn: ['Patient 1', 'Patient 2', 'Patient 3', 'OVERALL']


In [13]:
# ── Build formatted comparison table ─────────────────────────────────────────
def fmt(df):
    out = pd.DataFrame(index=df.index)
    out['IoU']     = df.apply(lambda r: f"{r['IoU_mean']:.3f} \u00b1 {r['IoU_std']:.3f}", axis=1)
    out['Dice']    = df.apply(lambda r: f"{r['Dice_mean']:.3f} \u00b1 {r['Dice_std']:.3f}", axis=1)
    out['HD (px)'] = df.apply(lambda r: f"{r['HD_mean_px']:.1f} \u00b1 {r['HD_std_px']:.1f}", axis=1)
    out['mAP']     = df['mAP'].map(lambda x: f"{float(x):.3f}")
    out['mAP@50']  = df['mAP@50'].map(lambda x: f"{float(x):.3f}")
    return out


models = {
    'SAM2 (raw)':           fmt(sam2_raw),
    'UltraSAM2 (raw)':      fmt(us2_raw),
    'SAM2 (denoised)':      fmt(sam2_dn),
    'UltraSAM2 (denoised)': fmt(us2_dn),
}

comparison = pd.concat(models, axis=1)
print(comparison.to_string())

              SAM2 (raw)                                           UltraSAM2 (raw)                                           SAM2 (denoised)                                           UltraSAM2 (denoised)                                          
                     IoU           Dice      HD (px)    mAP mAP@50             IoU           Dice      HD (px)    mAP mAP@50             IoU           Dice      HD (px)    mAP mAP@50                  IoU           Dice      HD (px)    mAP mAP@50
Patient 1  0.539 ± 0.102  0.695 ± 0.083  34.1 ± 12.1  0.155  0.643   0.414 ± 0.090  0.581 ± 0.087   31.3 ± 7.6  0.029  0.119   0.467 ± 0.111  0.630 ± 0.098  38.6 ± 12.5  0.079  0.309        0.413 ± 0.105  0.578 ± 0.103  31.4 ± 11.8  0.033  0.119
Patient 2  0.724 ± 0.084  0.837 ± 0.058  21.1 ± 17.1  0.492  1.000   0.702 ± 0.089  0.822 ± 0.062   13.9 ± 6.5  0.459  1.000   0.719 ± 0.077  0.834 ± 0.054   16.9 ± 9.8  0.492  1.000        0.625 ± 0.117  0.763 ± 0.095  18.0 ± 10.3  0.325  0.857
Patient 3  0.719

In [14]:
# ── Also build a numeric-only table for easier downstream processing ──────────
METRICS = ['IoU_mean', 'IoU_std', 'Dice_mean', 'Dice_std', 'HD_mean_px', 'HD_std_px', 'mAP', 'mAP@50']

numeric_models = {
    'SAM2 (raw)':           sam2_raw[METRICS],
    'UltraSAM2 (raw)':      us2_raw[METRICS],
    'SAM2 (denoised)':      sam2_dn[METRICS],
    'UltraSAM2 (denoised)': us2_dn[METRICS],
}

numeric = pd.concat(numeric_models, axis=1)
print(numeric.to_string())

          SAM2 (raw)                                                                 UltraSAM2 (raw)                                                                 SAM2 (denoised)                                                                 UltraSAM2 (denoised)                                                                
            IoU_mean IoU_std Dice_mean Dice_std HD_mean_px HD_std_px     mAP  mAP@50        IoU_mean IoU_std Dice_mean Dice_std HD_mean_px HD_std_px     mAP  mAP@50        IoU_mean IoU_std Dice_mean Dice_std HD_mean_px HD_std_px     mAP  mAP@50             IoU_mean IoU_std Dice_mean Dice_std HD_mean_px HD_std_px     mAP  mAP@50
Patient 1     0.5388  0.1020    0.6949   0.0834      34.09     12.08  0.1548  0.6429          0.4143  0.0900    0.5805   0.0875      31.29      7.59  0.0286  0.1190          0.4671  0.1106    0.6296   0.0977      38.61     12.49  0.0786  0.3095               0.4135  0.1049    0.5776   0.1026      31.36     11.80  0.0333  0.1190
Patient 2 

In [15]:
# ── Save ──────────────────────────────────────────────────────────────────────
from openpyxl.utils import get_column_letter

OUT_DIR = os.getcwd()

csv_path  = os.path.join(OUT_DIR, "sam2_vs_ultrasam2_comparison.csv")
xlsx_path = os.path.join(OUT_DIR, "sam2_vs_ultrasam2_comparison.xlsx")

comparison.to_csv(csv_path)

with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    comparison.to_excel(writer, sheet_name='Formatted')
    numeric.to_excel(writer,   sheet_name='Numeric')
    for sheet_name in writer.sheets:
        ws = writer.sheets[sheet_name]
        for col_idx in range(1, ws.max_column + 1):
            col_letter = get_column_letter(col_idx)
            max_len = max(
                (len(str(cell.value)) for cell in ws[col_letter] if cell.value is not None),
                default=8,
            )
            ws.column_dimensions[col_letter].width = max_len + 3

print(f"Saved CSV  → {csv_path}")
print(f"Saved XLSX → {xlsx_path}")

Saved CSV  → d:\graphs-n-tables\sam2_vs_ultrasam2_comparison.csv
Saved XLSX → d:\graphs-n-tables\sam2_vs_ultrasam2_comparison.xlsx
